# 🏗️ Thí nghiệm Kiến trúc – YOLOv8m vs YOLOv8s (Stage 1)

**Câu hỏi:** Nâng cấp từ YOLOv8s → YOLOv8m có cải thiện đáng kể Recall không? Và Trade-off Precision/Speed là bao nhiêu?

| | YOLOv8s (Baseline) | YOLOv8m (Thí nghiệm) |
|---|---|---|
| Tham số | ~11M | ~26M |
| Kiến trúc | Small | Medium |
| Conf (chuẩn) | 0.25 | 0.25 |
| Conf (tối ưu) | 0.15 | 0.15 |

**Kết quả YOLOv8s đã có (để so sánh):**
| Cấu hình | Precision | Recall | F1 |
|---|---|---|---|
| YOLOv8s conf=0.25 (Baseline) | 54.2% | 65.5% | 59.3% |
| YOLOv8s conf=0.15 (TN1_LowConf) | 40.8% | **68.6%** | 51.2% |

⚠️ **Yêu cầu:** GPU accelerator + Internet enabled

⏱️ **Ước tính thời gian:** ~45-60 phút train + ~10 phút đánh giá

In [ ]:
# ============================================================
# Cell 1: Clone repo + Cài đặt thư viện
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics

In [ ]:
# ============================================================
# Cell 2: Chuẩn bị dữ liệu
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"OK: {os.path.basename(task['src'])}")
    else:
        print(f"SKIP: {task['src']}")

print("\nDone.")

In [ ]:
# ============================================================
# Cell 3: Tiền xử lý dữ liệu
# ============================================================
%cd /kaggle/working/waste-detection2-Stage
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py
print("\n✅ Dữ liệu sẵn sàng!")

In [ ]:
# ============================================================
# Cell 4: Train YOLOv8m
# Cùng TOÀN BỘ siêu tham số với YOLOv8s gốc để so sánh công bằng
# ============================================================
from ultralytics import YOLO
from pathlib import Path
import time

PROJECT_DIR  = Path("/kaggle/working/waste-detection2-Stage")
DATASET_YAML = PROJECT_DIR / "data" / "processed_binary" / "dataset.yaml"
OUTPUT_DIR   = PROJECT_DIR / "results" / "yolo8m_runs"

print("=" * 60)
print("  🚀 Bắt đầu train YOLOv8m")
print("=" * 60)
print(f"  Dataset : {DATASET_YAML}")
print(f"  Output  : {OUTPUT_DIR}")
print()

model_m = YOLO("yolov8m.pt")  # Tải pretrained YOLOv8m từ COCO

t_start = time.time()

results_train = model_m.train(
    data        = str(DATASET_YAML),
    imgsz       = 640,
    epochs      = 100,
    batch       = 16,          # Cùng batch size với YOLOv8s
    patience    = 20,          # Cùng early stopping
    optimizer   = "auto",
    lr0         = 0.01,
    cos_lr      = True,        # Cosine Annealing scheduler
    augment     = True,
    workers     = 4,
    project     = str(OUTPUT_DIR),
    name        = "stage1_yolov8m",
    exist_ok    = True,
    save        = True,
    save_period = -1,          # Chỉ lưu best + last
    plots       = True,
    verbose     = True,
)

train_time = time.time() - t_start
print(f"\n✅ Train hoàn tất! Thời gian: {train_time/60:.1f} phút")

BEST_WEIGHTS_M = OUTPUT_DIR / "stage1_yolov8m" / "weights" / "best.pt"
print(f"   Best weights: {BEST_WEIGHTS_M}")

In [ ]:
# ============================================================
# Cell 5: Đánh giá YOLOv8m ở 2 mức conf (val set chính thức)
# ============================================================
from ultralytics import YOLO
from pathlib import Path

PROJECT_DIR  = Path("/kaggle/working/waste-detection2-Stage")
DATASET_YAML = PROJECT_DIR / "data" / "processed_binary" / "dataset.yaml"
BEST_WEIGHTS_M = PROJECT_DIR / "results" / "yolo8m_runs" / "stage1_yolov8m" / "weights" / "best.pt"

assert BEST_WEIGHTS_M.exists(), f"Không tìm thấy weights: {BEST_WEIGHTS_M}"

model_m = YOLO(str(BEST_WEIGHTS_M))

print("=" * 60)
print("  YOLOv8m – Validation conf=0.25 (chuẩn)")
print("=" * 60)
val_025 = model_m.val(
    data=str(DATASET_YAML), imgsz=640, batch=16,
    split="val", conf=0.25, verbose=True
)
print(f"  mAP50    : {val_025.box.map50:.4f}")
print(f"  mAP50-95 : {val_025.box.map:.4f}")
print(f"  Precision: {val_025.box.mp:.4f}")
print(f"  Recall   : {val_025.box.mr:.4f}")

print()
print("=" * 60)
print("  YOLOv8m – Validation conf=0.15 (hạ threshold)")
print("=" * 60)
val_015 = model_m.val(
    data=str(DATASET_YAML), imgsz=640, batch=16,
    split="val", conf=0.15, verbose=True
)
print(f"  mAP50    : {val_015.box.map50:.4f}")
print(f"  mAP50-95 : {val_015.box.map:.4f}")
print(f"  Precision: {val_015.box.mp:.4f}")
print(f"  Recall   : {val_015.box.mr:.4f}")

In [ ]:
# ============================================================
# Cell 6: Chạy inference tùy chỉnh (cùng metric Precision/Recall/F1
#         với cách tính trong experiment_stage1_inference.py)
#         → Đảm bảo so sánh apple-to-apple với bảng kết quả cũ
# ============================================================
import os, cv2, time, json
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR   = Path("/kaggle/working/waste-detection2-Stage")
VAL_IMG_DIR   = PROJECT_DIR / "data" / "processed_binary" / "images" / "val"
VAL_LBL_DIR   = PROJECT_DIR / "data" / "processed_binary" / "labels" / "val"
BEST_WEIGHTS_M = PROJECT_DIR / "results" / "yolo8m_runs" / "stage1_yolov8m" / "weights" / "best.pt"
OUTPUT_DIR    = PROJECT_DIR / "results" / "yolo8m_experiments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Hàm tính IoU & matching (giống hệt script cũ) ----------
def parse_yolo_label(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists(): return boxes
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cx, cy, w, h = map(float, parts[1:5])
            x1 = (cx - w/2) * img_w; y1 = (cy - h/2) * img_h
            x2 = (cx + w/2) * img_w; y2 = (cy + h/2) * img_h
            boxes.append([x1, y1, x2, y2])
    return boxes

def compute_iou(b1, b2):
    ix1=max(b1[0],b2[0]); iy1=max(b1[1],b2[1])
    ix2=min(b1[2],b2[2]); iy2=min(b1[3],b2[3])
    inter=max(0,ix2-ix1)*max(0,iy2-iy1)
    a1=(b1[2]-b1[0])*(b1[3]-b1[1]); a2=(b2[2]-b2[0])*(b2[3]-b2[1])
    union=a1+a2-inter
    return inter/union if union>0 else 0.0

def run_eval(model, val_images, conf_thresh):
    tp=fp=fn=0; total_gt=total_pred=0
    size_stats={"small":{"tp":0,"fn":0},"medium":{"tp":0,"fn":0},"large":{"tp":0,"fn":0}}
    for img_path in val_images:
        img=cv2.imread(str(img_path))
        if img is None: continue
        h,w=img.shape[:2]
        rel=img_path.relative_to(VAL_IMG_DIR)
        lbl=VAL_LBL_DIR/rel.with_suffix(".txt")
        gt_boxes=parse_yolo_label(lbl,w,h)
        results=model.predict(source=img,conf=conf_thresh,iou=0.7,verbose=False)
        pred_boxes=[]; pred_confs=[]
        for r in results:
            for box in r.boxes:
                pred_boxes.append(box.xyxy[0].cpu().numpy().tolist())
                pred_confs.append(float(box.conf[0]))
        # Greedy matching
        gt_matched=[False]*len(gt_boxes)
        ltp=lfp=0
        if pred_boxes:
            for idx in np.argsort(pred_confs)[::-1]:
                pb=pred_boxes[idx]; best_iou=0; best_gt=-1
                for j,gb in enumerate(gt_boxes):
                    if gt_matched[j]: continue
                    iou=compute_iou(pb,gb)
                    if iou>best_iou: best_iou=iou; best_gt=j
                if best_iou>=0.5 and best_gt>=0:
                    ltp+=1; gt_matched[best_gt]=True
                else: lfp+=1
        lfn=sum(1 for m in gt_matched if not m)
        tp+=ltp; fp+=lfp; fn+=lfn
        total_gt+=len(gt_boxes); total_pred+=len(pred_boxes)
        # Size analysis
        gm2=[False]*len(gt_boxes)
        if pred_boxes:
            for idx in np.argsort(pred_confs)[::-1]:
                pb=pred_boxes[idx]; bi=0; bgt=-1
                for j,gb in enumerate(gt_boxes):
                    if gm2[j]: continue
                    iou=compute_iou(pb,gb)
                    if iou>bi: bi=iou; bgt=j
                if bi>=0.5 and bgt>=0: gm2[bgt]=True
        for j,gb in enumerate(gt_boxes):
            bw=(gb[2]-gb[0])/w; bh=(gb[3]-gb[1])/h; ar=bw*bh
            cat="small" if ar<0.01 else ("medium" if ar<0.05 else "large")
            if gm2[j]: size_stats[cat]["tp"]+=1
            else: size_stats[cat]["fn"]+=1
    prec=tp/(tp+fp) if (tp+fp)>0 else 0
    rec=tp/(tp+fn) if (tp+fn)>0 else 0
    f1=2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    return {"Precision":round(prec,4),"Recall":round(rec,4),"F1":round(f1,4),
            "TP":tp,"FP":fp,"FN":fn,"Total GT":total_gt,"Total Pred":total_pred,
            "size_stats":size_stats}

# ---------- Chạy đánh giá ----------
model_m = YOLO(str(BEST_WEIGHTS_M))
exts = {".jpg",".jpeg",".png"}
val_images = sorted([p for p in VAL_IMG_DIR.rglob("*") if p.suffix.lower() in exts])
print(f"[INFO] Val images: {len(val_images)}")

configs = {
    "YOLOv8m_conf025": 0.25,
    "YOLOv8m_conf015": 0.15,
}

m_results = {}
for name, conf in configs.items():
    print(f"\n{'='*55}")
    print(f"  {name}  (conf={conf})")
    print(f"{'='*55}")
    t0=time.time()
    res=run_eval(model_m, val_images, conf)
    res["Time (s)"]=round(time.time()-t0,1)
    m_results[name]=res
    print(f"  Precision : {res['Precision']}")
    print(f"  Recall    : {res['Recall']}")
    print(f"  F1        : {res['F1']}")
    print(f"  TP={res['TP']}, FP={res['FP']}, FN={res['FN']}")
    print(f"  Thời gian : {res['Time (s)']}s")

In [ ]:
# ============================================================
# Cell 7: Bảng so sánh tổng hợp YOLOv8m vs YOLOv8s
# ============================================================
import pandas as pd
from IPython.display import display

# Kết quả YOLOv8s đã có từ thí nghiệm trước
yolov8s_results = {
    "YOLOv8s_conf025 (Baseline)": {"Model":"YOLOv8s","Conf":0.25,"Precision":0.5421,"Recall":0.6547,"F1":0.5931,"TP":476,"FP":402,"FN":251},
    "YOLOv8s_conf015 (TN1)": {"Model":"YOLOv8s","Conf":0.15,"Precision":0.4080,"Recall":0.6864,"F1":0.5118,"TP":499,"FP":724,"FN":228},
}

# Ghép kết quả YOLOv8m vào
rows = []
for name, d in yolov8s_results.items():
    rows.append({"Cấu hình": name, "Model": d["Model"], "Conf": d["Conf"],
                 "Precision": d["Precision"], "Recall": d["Recall"], "F1": d["F1"],
                 "TP": d["TP"], "FP": d["FP"], "FN": d["FN"]})

for name, d in m_results.items():
    rows.append({"Cấu hình": name, "Model": "YOLOv8m", "Conf": 0.25 if "025" in name else 0.15,
                 "Precision": d["Precision"], "Recall": d["Recall"], "F1": d["F1"],
                 "TP": d["TP"], "FP": d["FP"], "FN": d["FN"]})

df = pd.DataFrame(rows)

print("=" * 70)
print("  📊 BẢNG SO SÁNH TỔNG HỢP – YOLOv8m vs YOLOv8s")
print("=" * 70)
display(df.style.highlight_max(subset=["Recall","F1","Precision"], color="lightgreen")
              .highlight_min(subset=["FP","FN"], color="lightyellow"))

# In delta cải thiện
print("\n" + "=" * 70)
print("  📈 CẢI THIỆN CỦA YOLOv8m SO VỚI YOLOv8s (cùng conf)")
print("=" * 70)

base_025 = yolov8s_results["YOLOv8s_conf025 (Baseline)"]
base_015 = yolov8s_results["YOLOv8s_conf015 (TN1)"]
m_025 = m_results.get("YOLOv8m_conf025",{})
m_015 = m_results.get("YOLOv8m_conf015",{})

if m_025:
    print(f"  conf=0.25 → Recall: {base_025['Recall']:.4f} → {m_025['Recall']:.4f} (Δ = {m_025['Recall']-base_025['Recall']:+.4f})")
    print(f"  conf=0.25 → F1    : {base_025['F1']:.4f} → {m_025['F1']:.4f} (Δ = {m_025['F1']-base_025['F1']:+.4f})")
    print(f"  conf=0.25 → FP    : {base_025['FP']} → {m_025['FP']} (Δ = {m_025['FP']-base_025['FP']:+d})")
if m_015:
    print(f"  conf=0.15 → Recall: {base_015['Recall']:.4f} → {m_015['Recall']:.4f} (Δ = {m_015['Recall']-base_015['Recall']:+.4f})")
    print(f"  conf=0.15 → F1    : {base_015['F1']:.4f} → {m_015['F1']:.4f} (Δ = {m_015['F1']-base_015['F1']:+.4f})")
    print(f"  conf=0.15 → FP    : {base_015['FP']} → {m_015['FP']} (Δ = {m_015['FP']-base_015['FP']:+d})")

# Lưu CSV
df.to_csv(OUTPUT_DIR / "yolov8m_vs_s_comparison.csv", index=False, encoding="utf-8-sig")
print(f"\n[INFO] Đã lưu bảng so sánh: {OUTPUT_DIR / 'yolov8m_vs_s_comparison.csv'}")

In [ ]:
# ============================================================
# Cell 8: Vẽ biểu đồ so sánh + Training curves YOLOv8m
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from IPython.display import Image as IPImage, display

OUTPUT_DIR   = Path("/kaggle/working/waste-detection2-Stage/results/yolo8m_experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Biểu đồ 1: So sánh P/R/F1 ----
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("YOLOv8m vs YOLOv8s – Stage 1 Comparison", fontsize=15, fontweight="bold")

config_names = ["v8s\nconf=0.25", "v8s\nconf=0.15", "v8m\nconf=0.25", "v8m\nconf=0.15"]
prec_vals = [0.5421, 0.4080, m_results.get('YOLOv8m_conf025',{}).get('Precision',0), m_results.get('YOLOv8m_conf015',{}).get('Precision',0)]
rec_vals  = [0.6547, 0.6864, m_results.get('YOLOv8m_conf025',{}).get('Recall',0),    m_results.get('YOLOv8m_conf015',{}).get('Recall',0)]
f1_vals   = [0.5931, 0.5118, m_results.get('YOLOv8m_conf025',{}).get('F1',0),        m_results.get('YOLOv8m_conf015',{}).get('F1',0)]

import numpy as np
x = np.arange(len(config_names)); w = 0.25
ax = axes[0]
bars = [
    ax.bar(x-w, prec_vals, w, label="Precision", color="#4A90D9", alpha=0.85),
    ax.bar(x,   rec_vals,  w, label="Recall",    color="#7BC47F", alpha=0.85),
    ax.bar(x+w, f1_vals,   w, label="F1",        color="#E8943A", alpha=0.85),
]
for bs in bars:
    for b in bs:
        h=b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+0.01, f"{h:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(config_names, fontsize=10)
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1", fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# Thêm đường phân cách giữa v8s và v8m
ax.axvline(x=1.5, color="gray", linestyle="--", alpha=0.6, linewidth=1.5)
ax.text(0.5, 1.02, "YOLOv8s", ha="center", transform=ax.get_xaxis_transform(), fontsize=10, color="gray")
ax.text(2.5, 1.02, "YOLOv8m", ha="center", transform=ax.get_xaxis_transform(), fontsize=10, color="purple")

# ---- Biểu đồ 2: TP/FP/FN ----
ax = axes[1]
tp_vals = [476, 499, m_results.get('YOLOv8m_conf025',{}).get('TP',0), m_results.get('YOLOv8m_conf015',{}).get('TP',0)]
fp_vals = [402, 724, m_results.get('YOLOv8m_conf025',{}).get('FP',0), m_results.get('YOLOv8m_conf015',{}).get('FP',0)]
fn_vals = [251, 228, m_results.get('YOLOv8m_conf025',{}).get('FN',0), m_results.get('YOLOv8m_conf015',{}).get('FN',0)]

ax.bar(x, tp_vals, 0.6, label="TP (Đúng)",  color="#7BC47F", alpha=0.85)
ax.bar(x, fp_vals, 0.6, bottom=tp_vals, label="FP (Nhầm)",  color="#E8943A", alpha=0.85)
ax.bar(x, fn_vals, 0.6, bottom=[t+f for t,f in zip(tp_vals,fp_vals)], label="FN (Bỏ sót)", color="#D94A6B", alpha=0.85)
for i in range(len(config_names)):
    ax.text(i, tp_vals[i]/2, str(tp_vals[i]), ha="center", va="center", fontsize=10, fontweight="bold", color="white")
    ax.text(i, tp_vals[i]+fp_vals[i]/2, str(fp_vals[i]), ha="center", va="center", fontsize=10, fontweight="bold", color="white")
    ax.text(i, tp_vals[i]+fp_vals[i]+fn_vals[i]/2, str(fn_vals[i]), ha="center", va="center", fontsize=10, fontweight="bold", color="white")
ax.set_xticks(x); ax.set_xticklabels(config_names, fontsize=10)
ax.set_ylabel("Số lượng boxes"); ax.set_title("Phân phối TP / FP / FN", fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
ax.axvline(x=1.5, color="gray", linestyle="--", alpha=0.6, linewidth=1.5)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
chart_path = OUTPUT_DIR / "yolov8m_vs_s_comparison.png"
plt.savefig(str(chart_path), dpi=150, bbox_inches="tight")
plt.close()
print(f"[INFO] Đã lưu biểu đồ: {chart_path}")
display(IPImage(filename=str(chart_path), width=1100))

In [ ]:
# ============================================================
# Cell 9: Hiển thị Training Curves YOLOv8m
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image as IPImage, display

results_csv = Path("/kaggle/working/waste-detection2-Stage/results/yolo8m_runs/stage1_yolov8m/results.csv")

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("YOLOv8m Binary Detector – Training Curves", fontsize=14, fontweight="bold")
    
    plots = [
        (axes[0,0], "train/box_loss", "val/box_loss", "Box Loss"),
        (axes[0,1], "train/cls_loss", "val/cls_loss", "Classification Loss"),
        (axes[1,0], "metrics/mAP50(B)", "metrics/mAP50-95(B)", "mAP Metrics"),
        (axes[1,1], "metrics/precision(B)", "metrics/recall(B)", "Precision & Recall"),
    ]
    
    for ax, col1, col2, title in plots:
        label1 = col1.split("/")[-1].replace("(B)","")
        label2 = col2.split("/")[-1].replace("(B)","")
        if col1 in df.columns: ax.plot(df["epoch"], df[col1], label=label1, color="tab:blue")
        if col2 in df.columns: ax.plot(df["epoch"], df[col2], label=label2, color="tab:orange")
        ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    curve_path = Path("/kaggle/working/waste-detection2-Stage/results/yolo8m_experiments/yolov8m_training_curves.png")
    plt.savefig(str(curve_path), dpi=150, bbox_inches="tight")
    plt.close()
    display(IPImage(filename=str(curve_path), width=900))
else:
    print("[WARN] Không tìm thấy results.csv")

In [ ]:
# ============================================================
# Cell 10: Kết luận & Quyết định kiến trúc
# ============================================================
print("=" * 65)
print("  🏆 KẾT LUẬN THÍ NGHIỆM KIẾN TRÚC YOLOv8m vs YOLOv8s")
print("=" * 65)

m_025 = m_results.get('YOLOv8m_conf025', {})
m_015 = m_results.get('YOLOv8m_conf015', {})

if m_025 and m_015:
    # Recall cải thiện
    recall_gain_025 = m_025['Recall'] - 0.6547
    recall_gain_015 = m_015['Recall'] - 0.6864
    fp_change_025   = m_025['FP'] - 402
    fp_change_015   = m_015['FP'] - 724
    
    print(f"  YOLOv8m conf=0.25  → Recall: {m_025['Recall']:.4f}  (Δ = {recall_gain_025:+.4f} vs v8s)")
    print(f"                     → FP    : {m_025['FP']}  (Δ = {fp_change_025:+d} vs v8s)")
    print(f"  YOLOv8m conf=0.15  → Recall: {m_015['Recall']:.4f}  (Δ = {recall_gain_015:+.4f} vs v8s)")
    print(f"                     → FP    : {m_015['FP']}  (Δ = {fp_change_015:+d} vs v8s)")
    print()
    
    # Quyết định
    # Ưu tiên Recall cao nhất với FP thấp nhất
    best_recall = max(m_025['Recall'], m_015['Recall'], 0.6547, 0.6864)
    best_cfg = {
        m_025['Recall']: f"YOLOv8m conf=0.25  (Precision={m_025['Precision']:.4f})",
        m_015['Recall']: f"YOLOv8m conf=0.15  (Precision={m_015['Precision']:.4f})",
        0.6547: "YOLOv8s conf=0.25  (Precision=0.5421)",
        0.6864: "YOLOv8s conf=0.15  (Precision=0.4080)",
    }
    print(f"  🎯 Cấu hình Recall cao nhất: {best_cfg[best_recall]}")
    print(f"     Recall = {best_recall:.4f}")
    print()
    print("  💡 Trong hệ thống 2-Stage, Stage 1 ưu tiên RECALL cao.")
    print("     Vì Stage 2 (EfficientNet) sẽ lọc False Positives.")

print("=" * 65)

# Download
import shutil
shutil.make_archive("/kaggle/working/yolov8m_results", "zip",
                    "/kaggle/working/waste-detection2-Stage/results/yolo8m_experiments")
print("\n📦 Kết quả đã nén: /kaggle/working/yolov8m_results.zip")
print("   Bao gồm: biểu đồ so sánh, bảng CSV, training curves")
print("   → Download về máy để lưu trữ")